# 🎮 Asistente de Soporte para Steam
---

## 📝 ¿De qué trata este proyecto?
Este es un bot inteligente creado para ayudar a los jugadores con sus dudas en **Steam**. Este bot está entrenado para entender qué tipo de problema tienes y darte soluciones reales basadas en la potencia de tu computador.

---

## 🚀 ¿Qué puede hacer este bot?
* **Clasifica tus problemas:** Sabe si le hablas de un reembolso, un fallo técnico o un problema con tu cuenta.
* **Analiza tu PC:** Gracias a que lee datos en formato **JSON**, puede decirte si un juego te va a correr bien o si tu equipo necesita más potencia.
* **Tiene memoria:** No necesitas repetirle tu hardware en cada mensaje, el bot puede recordar lo que hablaron antes.
* **Responde al instante:** Puedes ver cómo escribe la respuesta palabra por palabra (Streaming), así no tienes que esperar a que termine de pensar todo el texto.

---

## 💻 PC de Prueba (Asus Vivobook X1404)
Para que el bot aprenda a dar buenos consejos, lo configuré con los datos de mi PC. Así él sabe exactamente qué recomendar para este equipo:
* **Modelo:** Asus Vivobook 14
* **Procesador:** Intel i5 (12va Generación)
* **Memoria RAM:** 8GB
* **Gráficos:** Intel Iris Xe
* **Sistema:** Windows 11

---


### 1. Preparación del Entorno
En esta sección instalamos las librerías necesarias y configuramos los motores de inteligencia artificial.

### Instalación de dependencias (Ejecutar solo una vez o si falta algún módulo)
```bash
pip install -U langchain-openai langchain-core python-dotenv langchain-text-splitters langchain-community faiss-cpu pandas
```

In [ ]:
import os
import json
import time
import pandas as pd 
from dotenv import load_dotenv

# Componentes de LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

# Componentes para RAG (Búsqueda Vectorial)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Carga de variables de entorno
load_dotenv()

# --- CONFIGURACIÓN DE MOTORES ---

# Modelo de Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url=os.getenv("GITHUB_BASE_URL")
)

# llm_logic: El "cerebro" analítico. Temperatura 0 para asegurar respuestas 
# exactas y consistentes durante la evaluación RAG.
llm_logic = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o-mini",
    temperature=0.0
)

# llm_chat: El motor conversacional. Con streaming activo para una 
# experiencia de usuario fluida.
llm_chat = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o-mini",
    streaming=True,
    temperature=0.7
)

print("✅ Motores y librerías inicializados. Listo para conectar la base de conocimiento.")

✅ Motores y librerías inicializados. Listo para conectar la base de conocimiento.


### 2. Configuración de Hardware (Contexto Técnico)
Para que el asistente de soporte brinde soluciones personalizadas, necesita conocer las especificaciones del equipo. Estos datos actúan como **contexto dinámico** que se suma a la información recuperada de la base de conocimientos.

El usuario puede optar por utilizar las especificaciones predeterminadas del **Asus Vivobook 14** o ingresar manualmente los datos de su propio computador.

In [9]:
# Hardware predeterminado para pruebas
hardware_ejemplo = {
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}

# Interacción para definir el contexto del hardware
print("¿Deseas ingresar los datos de tu propio hardware? (s/n)")
opcion = input().lower().strip()

if opcion == "s":
    print("\n--- Ingresa los datos de tu equipo ---")
    hardware_json = {
        "modelo": input("Modelo del equipo: "),
        "procesador": input("Procesador: "),
        "ram": input("Memoria RAM: "),
        "graficos": input("Tarjeta de video / Gráficos: "),
        "sistema": input("Sistema Operativo: ")
    }
else:
    print("\n✅ Usando hardware de prueba (Asus Vivobook 14).")
    hardware_json = hardware_ejemplo

# Guardamos el hardware en una variable global para el resto del bot
print("\n--- Hardware configurado para el soporte ---")
print(json.dumps(hardware_json, indent=4))

¿Deseas ingresar los datos de tu propio hardware? (s/n)

✅ Usando hardware de prueba (Asus Vivobook 14).

--- Hardware configurado para el soporte ---
{
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}


### 3. Implementación de la Base de Conocimiento (RAG)
En esta fase, transformamos la información textual en una **Base de Datos Vectorial**. Este proceso permite que el bot no dependa solo de su entrenamiento general, sino que consulte datos específicos y actualizados de Steam.

In [30]:
# 1. Base de conocimiento: Manual oficial de soporte de Steam
steam_knowledge = """
POLÍTICAS DE REEMBOLSO: 
- Juegos: Menos de 2 horas de juego y menos de 14 días desde la compra.
- DLC: Reembolsable si el juego base no se ha jugado más de 2 horas tras la compra del DLC.
- Preventas: Se puede pedir el reembolso en cualquier momento antes del lanzamiento.

JUEGOS DISPONIBLES EN STEAM Y SUS REQUISITOS:
- ELDEN RING: Mínimo 12GB RAM, i5-8400, GTX 1060 (6GB). Recomendado 16GB RAM.
- CYBERPUNK 2077: Mínimo 12GB RAM, GTX 1060 (6GB) y OBLIGATORIO instalar en SSD.
- COUNTER-STRIKE 2 (CS2): Mínimo 8GB RAM, procesador de 4 hilos, dedicada con 1GB (DirectX 11).
- STARDEW VALLEY: Muy ligero. 2GB RAM, GPU con 256MB de VRAM.
- BALDUR'S GATE 3: Mínimo 8GB RAM, GTX 970. Requiere SSD obligatoriamente.
- HELLDIVERS 2: Mínimo 8GB RAM, GTX 1050 Ti. Recomendado 16GB RAM.
- HADES II: Mínimo 4GB RAM, Dual Core 2.4 GHz, compatible con DirectX 12.
- RED DEAD REDEMPTION 2 (RDR2): Mínimo 8GB RAM, GTX 770. Recomendado 12GB RAM.
- FORZA HORIZON 5: Mínimo 8GB RAM, GTX 760. Recomendado 16GB RAM.
- APEX LEGENDS: Mínimo 6GB RAM, GT 640. Recomendado 8GB RAM.
- DOTA 2: Mínimo 4GB RAM, GPU compatible con DirectX 11.
- LETHAL COMPANY: Mínimo 4GB RAM, GTX 1050. Optimizado para laptops de estudio.
- TERRARIA: Mínimo 2GB RAM, 128MB VRAM. Muy ligero.
- DARK SOULS III: Mínimo 4GB RAM, Intel Core i3-2100, GTX 750 Ti.
- DARK SOULS REMASTERED: Mínimo 4GB RAM, Intel Core i5-2300, GTX 460.
- SEKIRO: SHADOWS DIE TWICE: Mínimo 4GB RAM, Intel Core i3-2100, GTX 760.
- HOLLOW KNIGHT: Mínimo 4GB RAM, Intel Core 2 Duo E8400.
- THE WITCHER 3: Mínimo 6GB RAM, Intel Core i5-2500K, GTX 660.
- GRAND THEFT AUTO V (GTA V): Mínimo 4GB RAM, 9800 GT 1GB.
- RUST: Mínimo 10GB RAM, Intel Core i7-3770. Exigente en RAM.
- 7 DAYS TO DIE: Mínimo 8GB RAM, 2.4 Ghz Dual Core, 2GB VRAM.
- LEFT 4 DEAD 2: Mínimo 2GB RAM, Pentium 4 3.0GHz.
- PORTAL 2: Mínimo 2GB RAM, Core 2 Duo E6600.
- TEAM FORTRESS 2: Mínimo 512MB RAM, procesador 1.7 GHz.
- VALHEIM: Mínimo 8GB RAM, Intel Dual Core 2.6 GHz, GTX 950.
- SEA OF THIEVES: Mínimo 4GB RAM, Intel Q9450, GTX 650.
- DESTINY 2: Mínimo 6GB RAM, Intel Core i3-3250, GTX 660 2GB.
- WARFRAME: Mínimo 4GB RAM, GPU compatible con DirectX 11.
- PATH OF EXILE: Mínimo 8GB RAM, Quad Core 2.6GHz, GTX 650 Ti.
- ARK: SURVIVAL EVOLVED: Mínimo 8GB RAM, Intel Core i5-2400, GTX 670 2GB.
- PHASMOPHOBIA: Mínimo 8GB RAM, Intel Core i5-4590, GTX 970.
- DEAD BY DAYLIGHT: Mínimo 8GB RAM, Intel Core i3-4170, GTX 460.
- ROCKET LEAGUE: Mínimo 4GB RAM, 2.5 GHz Dual Core, GTX 760.
- FALL GUYS: Mínimo 8GB RAM, Intel Core i5, GTX 660.
- AMONG US: Mínimo 1GB RAM, muy ligero.
- RIMWORLD: Mínimo 4GB RAM, Core 2 Duo.
- FACTORIO: Mínimo 4GB RAM, Dual Core 3GHz, 512MB VRAM.
- SLAY THE SPIRE: Mínimo 2GB RAM, procesador 2.0 GHz.
- VAMPIRE SURVIVORS: Mínimo 1GB RAM, soporte SSE2.
- DAVE THE DIVER: Mínimo 8GB RAM, Intel Core i3, GTX 750 Ti.
- MANOR LORDS: Mínimo 8GB RAM, Intel Core i5-4670, GTX 1050.
- PALWORLD: Mínimo 16GB RAM, i5-3570K, GTX 1050. Exigente en RAM.
- CONTENT WARNING: Mínimo 8GB RAM, GTX 1050 Ti.
- RESIDENT EVIL 4 REMAKE: Mínimo 8GB RAM, i5-7500, GTX 1050 Ti.
- STREET FIGHTER 6: Mínimo 8GB RAM, Intel Core i5-7500, GTX 1060.
- MONSTER HUNTER: WORLD: Mínimo 8GB RAM, Intel Core i5-4460, GTX 760.
- HORIZON ZERO DAWN: Mínimo 8GB RAM, GTX 780 (3GB).
- GOD OF WAR: Mínimo 8GB RAM, GTX 960 (4GB).
- SPIDER-MAN REMASTERED: Mínimo 8GB RAM, GTX 950.
- GHOST OF TSUSHIMA: Mínimo 8GB RAM, i3-7100, GTX 960 (4GB).
- EURO TRUCK SIMULATOR 2: Mínimo 4GB RAM, Dual Core 2.4 GHz.
- PROJECT ZOMBOID: Mínimo 8GB RAM, Quad-core 2.77GHz.

SOPORTE TÉCNICO Y ERRORES:
- Error 'Disco Corrupto': Se soluciona borrando la caché de descargas en Parámetros > Descargas.
- Juego no inicia: Verificar integridad de archivos o reinstalar DirectX y Visual C++.
- Steam Overlay: Revisar software de terceros como MSI Afterburner interfiriendo.
- Steam Deck: Juegos 'Verificado' funcionan perfecto; 'Jugable' puede requerir ajustes.

SEGURIDAD Y CUENTAS:
- Steam Guard (2FA): Obligatorio para habilitar intercambios inmediatos.
- Cuentas Bloqueadas: Puede ocurrir por 'Chargeback', intentos fallidos masivos o uso de VPN.
- Compartir Familia: Permite compartir con hasta 5 familiares. Algunos juegos no son compatibles.
- Guardado en la Nube: Sincroniza partidas guardadas de forma automática.

PAGOS Y CARTERA:
- Cargar Saldo: Se puede hacer mediante tarjetas de crédito, Redcompra (Webpay) o códigos de regalo de Steam.
- Error en Pago: Si la transacción falla, espera 30 minutos antes de reintentar para evitar bloqueos preventivos.
"""

# 2. Fragmentación (Chunking): Ajustamos el tamaño para no cortar requisitos a la mitad
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_text(steam_knowledge)

# 3. Creación de Base de Datos Vectorial con FAISS
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)

# Configuramos el recuperador para traer los 4 fragmentos más relevantes
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

print(f"✅ RAG configurado: {len(chunks)} chunks listos para búsqueda.")

✅ RAG configurado: 13 chunks listos para búsqueda.


### Revisar los pedazos de texto (CHUNKS)
Aquí vemos cómo quedó cortada la información de Steam. 
Sirve para chequear que los requisitos de los juegos se entiendan bien.

In [29]:
# 1. Visualización de fragmentos (Auditoría de texto)
print(f"🔍 Mostrando los {len(chunks)} chunks generados para el RAG:\n")

for i, chunk in enumerate(chunks):
    print(f"--- 📄 CHUNK {i+1} ---")
    print(chunk)
    print(f"📏 Tamaño: {len(chunk)} caracteres")
    print("-" * 45)

# 2. Verificación del estado de FAISS (Auditoría de Base de Datos)
print("\n📦 Estado de la Base de Datos Vectorial:")

try:
    # Accedemos al almacenamiento interno para contar los documentos indexados
    num_docs = len(vector_db.docstore._dict)
    print(f"✅ Éxito: Hay {num_docs} documentos listos para ser consultados por el retriever.")
    
    if num_docs == len(chunks):
        print("💎 Sincronización perfecta entre chunks y base de datos.")
    else:
        print("⚠️ Advertencia: Existe una discrepancia entre los chunks y la indexación.")
except Exception as e:
    print(f"❌ Error al verificar el docstore: {e}")

🔍 Mostrando los 13 chunks generados para el RAG:

--- 📄 CHUNK 1 ---
POLÍTICAS DE REEMBOLSO: 
- Juegos: Menos de 2 horas de juego y menos de 14 días desde la compra.
- DLC: Reembolsable si el juego base no se ha jugado más de 2 horas tras la compra del DLC.
- Preventas: Se puede pedir el reembolso en cualquier momento antes del lanzamiento.
📏 Tamaño: 273 caracteres
---------------------------------------------
--- 📄 CHUNK 2 ---
JUEGOS DISPONIBLES EN STEAM Y SUS REQUISITOS:
- ELDEN RING: Mínimo 12GB RAM, i5-8400, GTX 1060 (6GB). Recomendado 16GB RAM.
- CYBERPUNK 2077: Mínimo 12GB RAM, GTX 1060 (6GB) y OBLIGATORIO instalar en SSD.
- COUNTER-STRIKE 2 (CS2): Mínimo 8GB RAM, procesador de 4 hilos, dedicada con 1GB (DirectX 11).
- STARDEW VALLEY: Muy ligero. 2GB RAM, GPU con 256MB de VRAM.
📏 Tamaño: 362 caracteres
---------------------------------------------
--- 📄 CHUNK 3 ---
- BALDUR'S GATE 3: Mínimo 8GB RAM, GTX 970. Requiere SSD obligatoriamente.
- HELLDIVERS 2: Mínimo 8GB RAM, GTX 1050 T

### 4. El Juez: Auditoría de Fidelidad
Esta celda define la función que revisa si el bot está inventando información (alucinando) o si realmente está siguiendo el manual de Steam. 

**Objetivo:** Calificar la respuesta del 1 al 10 según qué tan fiel es al contexto recuperado.

In [23]:
def evaluar_fidelidad(pregunta, contexto, respuesta_bot):
    """
    Función auditora que utiliza llm_logic (Temp 0) para verificar 
    que no existan alucinaciones en la respuesta del asistente.
    """
    prompt_juez = f"""
    Eres un auditor de sistemas RAG extremadamente estricto y desconfiado. 
    Tu misión es verificar si la RESPUESTA DEL BOT es fiel ÚNICAMENTE al CONTEXTO oficial.

    REGLAS DE AUDITORÍA:
    1. Si el BOT menciona un juego, requisito o solucion que NO aparezca explícitamente en el CONTEXTO, el puntaje debe ser 1 (Alucinación).
    2. Si el BOT dice que no tiene información y verificas que el dato NO está en el CONTEXTO, califica con 10 (Honestidad).
    3. No importa si la respuesta es amable o útil; si contiene información externa al manual, es una falla de seguridad.
    4. No evalúes basándote en tu conocimiento en el mundo real. Si el juego existe pero NO fue entregado en el CONTEXTO, mencionarlo es una ALUCINACIÓN INACEPTABLE.
    5. Considera "Fiel" solo aquello que tenga respaldo textual en el contexto.

    PREGUNTA DEL USUARIO: {pregunta}
    CONTEXTO OFICIAL: {contexto}
    RESPUESTA DEL BOT: {respuesta_bot}

    Responde ÚNICAMENTE en formato JSON con la siguiente estructura:
    {{
        "puntaje": (int entre 0 y 10),
        "razon": (breve explicación del porqué de la nota)
    }}
    """
    
    try:
        # Usamos llm_logic con temperatura 0 para asegurar una evaluación determinista
        resultado = llm_logic.invoke(prompt_juez)
        
        # Limpieza de posibles formatos Markdown en la respuesta del LLM
        json_clean = resultado.content.replace("```json", "").replace("```", "").strip()
        return json.loads(json_clean)
        
    except Exception as e:
        return {
            "puntaje": 0, 
            "razon": f"Error técnico en la fase de auditoría: {str(e)}"
        }

print("⚖️ Juez de Fidelidad cargado y listo para supervisar el chat.")

⚖️ Juez de Fidelidad cargado y listo para supervisar el chat.


### 🧠 5. Clasificación y Razonamiento Universal (Lógica con RAG)
En esta sección definimos el "cerebro" del asistente. La función realiza tres tareas críticas:

1.  **Clasificación (Few-Shot):** Identifica la intención del usuario, incluyendo la nueva categoría **[RECOMENDACION]**.
2.  **Recuperación Dinámica (RAG):** Ajusta el alcance de la búsqueda. Si es una recomendación, recupera **15 fragmentos** para evaluar más opciones; para soporte técnico, mantiene **4 fragmentos** para mayor precisión.
3.  **Razonamiento Técnico (Chain of Thought):** Contrasta los requisitos oficiales con el hardware del usuario.

**Regla de Oro:** Se prohíbe el uso de conocimiento externo. Si un juego no está en el manual de la Celda 3, el bot debe admitir que no tiene información oficial.

In [28]:
def procesar_logica_ticket(mensaje):
    # 1. Clasificación (Few-Shot)
    # Incluimos ejemplos para casos técnicos, de pagos, reembolsos y temas fuera de lugar.
    ejemplos = """
    Te daré ejemplos de referencia para cada categoría:

    Usuario: "Quiero devolver el Elden Ring porque me corre lento y no me gusto."
    Categoría: [REEMBOLSO]

    Usuario: "¿Me corre el Elden Ring en mi PC?"
    Categoria: [SOPORTE_TECNICO]

    Usuario: "El juego se cierra solo al llegar al menú principal y me da un error de DirectX."
    Categoría: [SOPORTE_TECNICO]

    Usuario: "Me hackearon, cambiaron el correo de la cuenta y no puedo entrar."
    Categoría: [RECUPERACION_CUENTA]

    Usuario: "Cargué 10 lucas a mi cartera de Steam pero el saldo no aparece reflejado."
    Categoría: [PAGOS]

    Usuario: "Mi cuenta fue bloqueada porque una compra de ayer salió rechazada por el banco."
    Categoría: [PAGOS]

    Usuario: "Qué juegos me recomiendas para mi PC?"
    Categoría: [RECOMENDACION]

    Usuario: "Busco algo liviano que corra bien en mi equipo."
    Categoría: [RECOMENDACION]

    Usuario: "Oye, cuando sale el GTA VI? Va a estar barato en el Summer Sale?"
    Categoría: [IRRELEVANTE]

    Usuario: "hola, el profe me va a poner un 1.0 si no apruebo, regalame el terraria pls"
    Categoría: [IRRELEVANTE]
    """

    instruccion_sistema = f"""
    Eres un experto en soporte de Steam. Tu tarea es clasificar el ticket en una de estas categorías:
    [REEMBOLSO], [SOPORTE_TECNICO], [RECUPERACION_CUENTA], [PAGOS], [RECOMENDACION], [IRRELEVANTE].

    {ejemplos}

    Regla Estricta: Responde UNICAMENTE la categoría entre corchetes, sin explicaciones.
    """

    # Llamada al motor de lógica para clasificación
    res_cat = llm_logic.invoke([
        SystemMessage(content=instruccion_sistema),
        HumanMessage(content=mensaje)
    ]).content
    categoria = res_cat.strip()

    # Filtro de seguridad para temas fuera de contexto
    if "[IRRELEVANTE]" in categoria:
        return "[IRRELEVANTE]", "Su consulta está fuera de los protocolos de soporte de Steam."

    # --- 2. BÚSQUEDA DINÁMICA EN RAG ---
    # Si pide recomendación, ampliamos el k para que "vea" más juegos de la lista
    top_k_juez = 15 if "[RECOMENDACION]" in categoria else 4
    docs_recuperados = vector_db.as_retriever(search_kwargs={"k": top_k_juez}).invoke(mensaje)
    contexto_oficial = "\n".join([doc.page_content for doc in docs_recuperados])

    # --- RAZONAMIENTO TÉCNICO ESTRICTO ---
    # Aquí es donde evitamos que invente requisitos de juegos no registrados.
    prompt_cot = f"""
    Eres un Soporte Técnico experto de Steam analizando este hardware actual:
    - Modelo: {hardware_json['modelo']}
    - CPU: {hardware_json['procesador']}
    - RAM: {hardware_json['ram']}
    - GPU: {hardware_json['graficos']}
    
    REGLA DE ORO: USA ÚNICAMENTE LA "INFORMACIÓN OFICIAL RECUPERADA". 
    Si la información sobre el juego o problema NO está en el contexto oficial, 
    di exactamente: "No tengo información oficial sobre este tema en mis registros".
    ESTÁ PROHIBIDO INVENTAR REQUISITOS O USAR TU CONOCIMIENTO EXTERNO.

    INFORMACIÓN OFICIAL RECUPERADA: 
    {contexto_oficial}

    TAREAS:
    1. Si la categoría es [RECOMENDACION]: Sugiere juegos del contexto que el hardware anterior pueda ejecutar bien.

    Analiza paso a paso:
    1. ANALIZAR: ¿La información solicitada existe en el contexto oficial?
    2. EVALUAR: Si existe, ¿cómo se compara con las specs del usuario?
    3. DETERMINAR: Entrega una respuesta técnica, honesta y basada 100% en el contexto.

    Ticket del usuario: "{mensaje}"
    """
    
    analisis = llm_logic.invoke([
        SystemMessage(content=prompt_cot),
        HumanMessage(content=mensaje)
    ]).content
    
    return categoria, analisis

print("⚙️ Lógica de procesamiento y filtro de Groundedness configurados.")

⚙️ Lógica de procesamiento y filtro de Groundedness configurados.


### 🎮 6. Chatbot Interactivo con Auditoría en Tiempo Real
Esta es la interfaz final del asistente. Integra todos los módulos anteriores en un bucle de conversación continuo.

**Características de esta implementación:**
* **Memoria de Corto Plazo:** Utiliza `InMemoryChatMessageHistory` para recordar el contexto de la conversación actual.
* **Transparencia de Fidelidad:** El usuario ve en tiempo real el puntaje otorgado por el "Juez", lo que genera confianza en las respuestas técnicas.
* **Prevención de Alucinaciones:** El motor de chat es forzado a utilizar el análisis técnico validado por la lógica interna, evitando que invente datos sobre juegos no registrados.
* **Experiencia Fluida:** Implementa *Streaming* para visualizar la respuesta palabra por palabra.

In [ ]:
# Historial de mensajes en memoria
history = InMemoryChatMessageHistory()

def iniciar_chat_steam():
    print(f"=== 🎮 ASISTENTE DE STEAM ({hardware_json['modelo']}) ===")
    print("Escribe 'salir' para terminar la conversación.\n")
    
    # 1. Configuración del mensaje de sistema inicial
    if len(history.messages) == 0:
        msg_inicial = (
            f"Eres el soporte de Steam para un usuario con {hardware_json['modelo']}. "
            f"Especificaciones del equipo: {json.dumps(hardware_json)}. "
            "Tu tono es profesional y técnico. Si no tienes información oficial, admítelo."
        )
        history.add_message(SystemMessage(content=msg_inicial))

    while True:
        user_input = input("\n🧑 Tú: ")
        if user_input.lower() in ['salir', 'exit', 'quit']: 
            print("👋 ¡Hasta luego!")
            break
        if not user_input:
            continue

        print("🔍 Procesando consulta...", end="\r")

        # 2. Ejecutar Lógica y RAG (Obtenemos el análisis técnico)
        categoria, analisis_tecnico = procesar_logica_ticket(user_input)
        
        # 3. Auditoría en tiempo real (El Juez)
        puntaje_info = ""
        if "[IRRELEVANTE]" not in categoria:
            # Sincronizamos al Juez con el Cerebro
            top_k_juez = 15 if "[RECOMENDACION]" in categoria else 4
            
            # USAMOS la variable aquí para buscar el contexto que el Juez va a revisar
            docs = vector_db.as_retriever(search_kwargs={"k": top_k_juez}).invoke(user_input)
            contexto_usado = "\n".join([doc.page_content for doc in docs])
            
            # El juez evalúa el análisis técnico generado
            evaluacion = evaluar_fidelidad(user_input, contexto_usado, analisis_tecnico)
            puntaje = evaluacion.get('puntaje', 0)
            mood = "✅" if puntaje >= 8 else "⚠️" if puntaje >= 5 else "❌"
            puntaje_info = f" | [Fidelidad: {puntaje}/10 {mood}]"
        
        # Limpiar el mensaje de "Procesando" y mostrar encabezado
        print(" " * 50, end="\r")
        print(f"🤖 Soporte [{categoria}]{puntaje_info}: ", end="", flush=True)
        
        # Caso A: El tema es ajeno a Steam
        if "[IRRELEVANTE]" in categoria:
            print(analisis_tecnico) 
            continue
        
        # Caso B: Soporte Técnico / Pagos / Reembolsos (Streaming)
        # Forzamos al Chat a usar el análisis aprobado por la lógica y el juez
        prompt_final = [
            *history.messages,
            HumanMessage(content=(
                f"Sigue estrictamente este análisis técnico: {analisis_tecnico}\n\n"
                f"Pregunta del usuario: {user_input}"
            ))
        ]
        
        full_ai_response = ""
        for chunk in llm_chat.stream(prompt_final):
            content = chunk.content
            print(content, end="", flush=True)
            full_ai_response += content
            time.sleep(0.01) # Pequeño delay para suavizar el streaming
            
        # Actualizamos el historial con la interacción validada
        history.add_user_message(user_input)
        history.add_ai_message(full_ai_response)
        print() # Salto de línea final

# Lanzar el chatbot
iniciar_chat_steam()

=== 🎮 ASISTENTE DE STEAM (Asus Vivobook 14) ===
Escribe 'salir' para terminar la conversación.

🤖 Soporte [[RECOMENDACION]] | [Fidelidad: 1/10 ❌]: Con base en las especificaciones de tu Asus Vivobook 14, que cuenta con un procesador Intel i5-1235U, 8GB de RAM y gráficos Intel Iris Xe, aquí tienes cinco juegos que deberías poder correr sin problemas:

1. **Hades II**: Este juego exige mínimo 4GB de RAM y un procesador Dual Core a 2.4 GHz, así que tu equipo cumple con los requisitos.

2. **Hollow Knight**: Con un requerimiento de 4GB de RAM y un procesador Intel Core 2 Duo E8400, tu configuración también es adecuada para este juego.

3. **Dota 2**: Requiere 4GB de RAM y una GPU compatible con DirectX 11. Dado que tu Intel Iris Xe es compatible con DirectX 12, podrás jugar sin inconvenientes.

4. **Valheim**: Este juego tiene un requisito mínimo de 8GB de RAM y un procesador Intel Dual Core a 2.6 GHz. Aunque tu procesador es más moderno, la exigencia de RAM es cumplida.

5. **Phasmophobia